# DINOv3 ViT-L/16 Embeddings with an MLP Classifier

## Supervised culvert-blockage classification using frozen visual representations

This notebook implements the supervised DINOv3 + MLP experiment used to examine
a **frozen pretrained visual representation** can support classification
of culvert trash-screen conditions at previously unseen monitoring sites.

Each CCTV image has already been converted into a DINOv3 ViT-L/16 CLS
embedding. DINOv3 is used here as a fixed feature extractor: its
parameters are not updated in this notebook. A comparatively small
multi-layer perceptron (MLP) is trained on the extracted feature vectors to
distinguish **clear** from **blocked** conditions.

### Experimental structure

The ten monitoring sites are divided into:

- **8 development sites**, used for MLP fitting, hyperparameter selection and
  decision-threshold selection.
- **2 held-out test sites**, Cornwall Bude Cedar Grove and Brutondam, which are
  excluded from model development and used only for final cross-site
  evaluation.

Within the eight development sites, the original experiment uses a
**stratified image-level 85/15 fit-validation split**. This means that all
development sites can contribute images to both subsets. The split preserves
the class balance and is reproducible through a fixed random seed. This is
distinct from the final cross-site test, where both test cameras remain fully
unseen during model development.

### Reproducibility

The public version uses repository-relative paths and contains no machine- or
Google Drive-specific directories. Expected inputs are:

```text
data/
├── site_split.csv
└── dinov3_vitl16_cls_embeddings/
    ├── dinov3_vitl16_cls_<site>.npy
    └── meta_<site>.csv
```

Generated artefacts are written to:

```text
outputs/dinov3_vitl16_mlp/
```

Saved notebook outputs have been cleared so that the repository does not expose
local paths or cached execution state.


In [ ]:
# Reproducible experiment setup

import copy
import itertools
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)

RANDOM_SEED = 42


def set_seed(seed=RANDOM_SEED):
    """Set random seeds used by Python, NumPy and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # These settings favour reproducibility over maximum GPU throughput.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

# Repository-relative paths.
# The notebook assumes it is launched from the repository root.
REPO_ROOT = Path.cwd()
DATA_ROOT = REPO_ROOT / "data"

DINOV3_FOLDER = DATA_ROOT / "dinov3_vitl16_cls_embeddings"
SPLIT_PATH = DATA_ROOT / "site_split.csv"
RESULTS_FOLDER = REPO_ROOT / "outputs" / "dinov3_vitl16_mlp"

RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"Using device: {device}")
print(f"Embedding directory: {DINOV3_FOLDER}")
print(f"Results folder: {RESULTS_FOLDER}")


In [ ]:
# Load precomputed DINOv3 ViT-L/16 CLS embeddings and metadata

sites = [
    "sites_corshamaqueduct_cam1",
    "Cornwall_BudeCedarGrove",
    "Devon_BarnstapleConeyGut_Scree",
    "Cornwall_Mevagissey_PreScree",
    "Cornwall_PenzanceCC",
    "Devon_Buckfastleigh",
    "sites_pilloutfall_cam1",
    "Cornwall_KingsandCP",
    "Devon_LympstoneScree",
    "sites_brutondam_cam1"
]

all_dino, all_meta = [], []

for site in sites:
    embedding_path = DINOV3_FOLDER / f"dinov3_vitl16_cls_{site}.npy"
    metadata_path = DINOV3_FOLDER / f"meta_{site}.csv"

    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Missing embedding file for {site}: {embedding_path}"
        )

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Missing metadata file for {site}: {metadata_path}"
        )

    all_dino.append(np.load(embedding_path))
    all_meta.append(pd.read_csv(metadata_path))


embeddings = np.vstack(all_dino)

metadata = pd.concat(
    all_meta,
    ignore_index=True
)

assert len(embeddings) == len(metadata)
assert {"site", "label"}.issubset(metadata.columns)
assert set(metadata["label"].str.lower().unique()).issubset(
    {"clear", "blocked"}
)

print(
    f"Loaded {embeddings.shape[0]} embeddings, "
    f"dim={embeddings.shape[1]}"
)

print(metadata["label"].value_counts())


In [ ]:
# Basic input-data checks

print(f"Number of monitoring sites loaded: {metadata['site'].nunique()}")
print("\nImages per site:")
display(
    metadata.groupby(["site", "label"])
    .size()
    .unstack(fill_value=0)
)


In [ ]:
# Define the cross-site development/test split

if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Site split file not found: {SPLIT_PATH}"
    )

split_summary = pd.read_csv(SPLIT_PATH)

required_columns = {"site", "role"}
if not required_columns.issubset(split_summary.columns):
    raise ValueError(
        f"site_split.csv must contain columns: {sorted(required_columns)}"
    )

ALL_SITES = sorted(split_summary["site"].unique())

TEST_SITES = sorted(
    split_summary.loc[
        split_summary["role"].str.lower() == "test",
        "site"
    ].tolist()
)

DEVELOPMENT_SITES = sorted(
    split_summary.loc[
        split_summary["role"].str.lower() == "development",
        "site"
    ].tolist()
)

EXPECTED_TEST_SITES = {
    "Cornwall_BudeCedarGrove",
    "sites_brutondam_cam1"
}

if len(ALL_SITES) != 10:
    raise ValueError(f"Expected 10 sites but found {len(ALL_SITES)}.")

if len(DEVELOPMENT_SITES) != 8:
    raise ValueError(
        f"Expected 8 development sites but found {len(DEVELOPMENT_SITES)}."
    )

if set(TEST_SITES) != EXPECTED_TEST_SITES:
    raise ValueError(
        "Unexpected held-out test sites. "
        f"Expected {sorted(EXPECTED_TEST_SITES)}, found {TEST_SITES}."
    )

print("Development sites:")
for site in DEVELOPMENT_SITES:
    print(f"  {site}")

print("\nHeld-out test sites:")
for site in TEST_SITES:
    print(f"  {site}")


## 1. Construct development and held-out datasets

The site split is applied before any MLP training. Images from the two test
sites are extracted into a separate dataset and are not used to select network
architecture, regularisation parameters, stopping epoch or decision threshold.

Labels are encoded as `0 = clear` and `1 = blocked`, making the blockage class
the positive class for ROC-AUC, precision, recall and F1 calculations.


In [ ]:
# Heldout test dataset

def extract_sites(selected_sites):
    mask = metadata["site"].isin(selected_sites)

    X = embeddings[mask.to_numpy()]
    meta = metadata.loc[mask].reset_index(drop=True)

    y = (
        meta["label"]
        .str.lower()
        .eq("blocked")
        .astype(np.int64)
        .to_numpy()
    )

    return X, y, meta


X_development, y_development, development_meta = extract_sites(DEVELOPMENT_SITES)
X_test, y_test, test_meta = extract_sites(TEST_SITES)

print(f"Development images: {len(X_development)}")
print(f"Test images: {len(X_test)}")

print("\nDevelopment classes:")
print(pd.Series(y_development).value_counts().sort_index())

print("\nTest classes:")
print(pd.Series(y_test).value_counts().sort_index())


## 2. Create the internal development validation split

The eight development sites are divided into fitting and validation images
using a stratified 85/15 random split. Stratification maintains the class
balance in both subsets.

The validation subset serves two purposes:

1. selecting the MLP hyperparameter configuration using ROC-AUC; and
2. selecting the final classification threshold by maximising validation F1.

Because this split is performed at image level, the validation result measures
generalisation to unseen images from the development cameras, not to unseen
camera sites. Cross-site generalisation is assessed separately using the two
held-out test sites.


In [ ]:
# Validation split

development_indices = np.arange(len(X_development))

fit_indices, validation_indices = train_test_split(
    development_indices,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=y_development
)

X_fit = X_development[fit_indices]
y_fit = y_development[fit_indices]

X_validation = X_development[validation_indices]
y_validation = y_development[validation_indices]

fit_meta = development_meta.iloc[fit_indices].reset_index(drop=True)
validation_meta = development_meta.iloc[validation_indices].reset_index(drop=True)

print(f"Fit images: {len(X_fit)}")
print(f"Validation images: {len(X_validation)}")

print("\nFit class distribution:")
print(pd.Series(y_fit).value_counts().sort_index())

print("\nValidation class distribution:")
print(pd.Series(y_validation).value_counts().sort_index())

print("\nValidation sites:")
print(validation_meta["site"].value_counts())


## 3. Define the MLP classifier and training procedure

The input to the classifier is the fixed DINOv3 CLS embedding rather than raw
pixels. The MLP consists of one or two fully connected hidden layers, ReLU
activations and dropout regularisation, followed by a single output logit.

Training uses binary cross-entropy with logits and the Adam optimiser. Early
stopping monitors validation ROC-AUC, retaining the model state from the epoch
with the highest validation AUC. This prevents the final model selection from
depending on the held-out test sites.


In [ ]:
# DataLoader function

def create_loader(X, y, batch_size=64, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)

    dataset = TensorDataset(X_tensor, y_tensor)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return loader


In [ ]:
# MLP classifier

class MLPClassifier(nn.Module):

    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))

            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        logits = self.network(x)
        return logits.squeeze(1)


In [ ]:
# Prediction function

def predict_scores(model, loader):
    model.eval()

    all_scores = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            scores = torch.sigmoid(logits)

            all_scores.append(scores.cpu().numpy())
            all_targets.append(y_batch.numpy())

    all_scores = np.concatenate(all_scores)
    all_targets = np.concatenate(all_targets)

    return all_scores, all_targets


In [ ]:
# Training with early stopping

def train_mlp(
    X_fit,
    y_fit,
    X_validation,
    y_validation,
    hidden_dims,
    dropout,
    learning_rate,
    weight_decay,
    batch_size=64,
    max_epochs=100,
    patience=8,
    seed=RANDOM_SEED
):
    set_seed(seed)

    fit_loader = create_loader(X_fit, y_fit, batch_size=batch_size, shuffle=True)
    validation_loader = create_loader(X_validation, y_validation, batch_size=batch_size, shuffle=False)

    model = MLPClassifier(
        input_dim=X_fit.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.BCEWithLogitsLoss()

    best_validation_auc = float("-inf")
    best_epoch = 0
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(1, max_epochs + 1):

        model.train()
        total_loss = 0.0

        for X_batch, y_batch in fit_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)

        mean_training_loss = total_loss / len(fit_loader.dataset)

        validation_scores, validation_targets = predict_scores(model, validation_loader)

        validation_auc = roc_auc_score(validation_targets, validation_scores)

        history.append({
            "epoch": epoch,
            "training_loss": mean_training_loss,
            "validation_auc": validation_auc
        })

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is None:
        raise RuntimeError("No valid model state was recorded.")

    model.load_state_dict(best_state)

    return {
        "model": model,
        "best_validation_auc": best_validation_auc,
        "best_epoch": best_epoch,
        "history": pd.DataFrame(history)
    }


## 4. Hyperparameter selection

A compact grid search evaluates alternative hidden-layer structures, dropout
rates, learning rates and weight-decay values. Each candidate is trained with
the same random seed and assessed on the same development validation split.

The configuration with the highest validation ROC-AUC is selected for
subsequent threshold selection and held-out evaluation. ROC-AUC is appropriate
at this stage because it assesses ranking performance independently of a fixed
classification threshold.


In [ ]:
# Hyperparameter grid

parameter_grid = {
    "hidden_dims": [[128], [256, 128], [512, 128]],
    "dropout": [0.2, 0.4],
    "learning_rate": [1e-4, 1e-3],
    "weight_decay": [0.0, 1e-4]
}

parameter_combinations = list(itertools.product(
    parameter_grid["hidden_dims"],
    parameter_grid["dropout"],
    parameter_grid["learning_rate"],
    parameter_grid["weight_decay"]
))

print(f"Number of configurations: {len(parameter_combinations)}")


In [ ]:
# Run grid search

grid_results = []
trained_models = {}

start_time = time.time()

for configuration_number, configuration in enumerate(parameter_combinations, start=1):
    hidden_dims, dropout, learning_rate, weight_decay = configuration

    print(f"\nConfiguration {configuration_number} of {len(parameter_combinations)}")
    print(f"Hidden dimensions: {hidden_dims}, dropout: {dropout}, learning rate: {learning_rate}, weight decay: {weight_decay}")

    training_result = train_mlp(
        X_fit=X_fit,
        y_fit=y_fit,
        X_validation=X_validation,
        y_validation=y_validation,
        hidden_dims=hidden_dims,
        dropout=dropout,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        batch_size=64,
        max_epochs=100,
        patience=8,
        seed=RANDOM_SEED
    )

    trained_models[configuration_number] = {
        "model": training_result["model"],
        "history": training_result["history"]
    }

    grid_results.append({
        "configuration": configuration_number,
        "hidden_dims": str(hidden_dims),
        "dropout": dropout,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "best_epoch": training_result["best_epoch"],
        "validation_auc": training_result["best_validation_auc"]
    })

grid_results_df = pd.DataFrame(grid_results)
grid_results_df = grid_results_df.sort_values("validation_auc", ascending=False).reset_index(drop=True)

elapsed_minutes = (time.time() - start_time) / 60
print(f"\nGrid search completed in {elapsed_minutes:.2f} minutes")

display(grid_results_df)


In [ ]:
# Select best configuration

best_result = grid_results_df.iloc[0]

best_configuration_number = int(best_result["configuration"])

best_model = trained_models[best_configuration_number]["model"]
best_history = trained_models[best_configuration_number]["history"]

print("Selected configuration:")
print(best_result)

print("\nSelected model:")
print(best_model)


In [ ]:
# Inspect selected model training history

plt.figure(figsize=(7, 5))
plt.plot(best_history["epoch"], best_history["training_loss"])
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Selected MLP training loss")
plt.tight_layout()
plt.show()


plt.figure(figsize=(7, 5))
plt.plot(best_history["epoch"], best_history["validation_auc"])
plt.xlabel("Epoch")
plt.ylabel("Validation AUC")
plt.title("Selected MLP validation AUC")
plt.tight_layout()
plt.show()


## 5. Select the operating threshold

The MLP returns a continuous blockage probability after the sigmoid
transformation. A binary operating threshold is therefore required to assign
each image to the clear or blocked class.

The threshold is selected only from the development validation set. Candidate
thresholds from the precision-recall curve are evaluated and the threshold that
maximises validation F1 is fixed before the held-out test data are scored.
This avoids optimistically tuning the decision rule on the final test sites.


In [ ]:
# Select threshold using validation data

validation_loader = create_loader(X_validation, y_validation, batch_size=64, shuffle=False)

validation_scores, validation_targets = predict_scores(best_model, validation_loader)

validation_auc = roc_auc_score(validation_targets, validation_scores)

precision_values, recall_values, thresholds = precision_recall_curve(validation_targets, validation_scores)

f1_values = 2 * precision_values * recall_values / (precision_values + recall_values + 1e-12)

valid_f1_values = f1_values[:-1]
best_threshold_index = np.argmax(valid_f1_values)

selected_threshold = thresholds[best_threshold_index]
selected_validation_f1 = valid_f1_values[best_threshold_index]

print(f"Validation AUC: {validation_auc:.4f}")
print(f"Selected validation threshold: {selected_threshold:.6f}")
print(f"Validation F1 at selected threshold: {selected_validation_f1:.4f}")


## 6. Evaluate the two unseen monitoring sites

The selected MLP and fixed validation-derived threshold are now applied to
Cornwall Bude Cedar Grove and Brutondam. These sites were excluded from model
fitting and model selection.

Performance is reported both after pooling the two sites and separately for
each site. The pooled result summarises overall classification performance,
whereas the per-site results expose differences in transfer between camera
environments that could be hidden by a single aggregate metric.


In [ ]:
# Evaluate held out test sites

test_loader = create_loader(X_test, y_test, batch_size=64, shuffle=False)

test_scores, test_targets = predict_scores(best_model, test_loader)

test_predictions = (test_scores >= selected_threshold).astype(int)

overall_test_auc = roc_auc_score(test_targets, test_scores)
overall_test_accuracy = accuracy_score(test_targets, test_predictions)
overall_test_precision = precision_score(test_targets, test_predictions, zero_division=0)
overall_test_recall = recall_score(test_targets, test_predictions, zero_division=0)
overall_test_f1 = f1_score(test_targets, test_predictions, zero_division=0)

print("Pooled held out test results")
print(f"AUC: {overall_test_auc:.4f}")
print(f"Accuracy: {overall_test_accuracy:.4f}")
print(f"Precision: {overall_test_precision:.4f}")
print(f"Recall: {overall_test_recall:.4f}")
print(f"F1: {overall_test_f1:.4f}")
print(f"Threshold selected using validation data: {selected_threshold:.6f}")


In [ ]:
# Add predictions to test metadata

test_results = test_meta.copy()

test_results["y_true"] = test_targets.astype(int)
test_results["blockage_score"] = test_scores
test_results["predicted_class"] = test_predictions

test_results["true_label"] = np.where(test_results["y_true"] == 1, "blocked", "clear")
test_results["predicted_label"] = np.where(test_results["predicted_class"] == 1, "blocked", "clear")

display(test_results.head())


In [ ]:
# Calculate per site AUC and metrics

per_site_results = []

for site in TEST_SITES:
    site_results = test_results[test_results["site"] == site].copy()

    y_site = site_results["y_true"].to_numpy()
    score_site = site_results["blockage_score"].to_numpy()
    prediction_site = site_results["predicted_class"].to_numpy()

    site_auc = roc_auc_score(y_site, score_site)
    site_accuracy = accuracy_score(y_site, prediction_site)
    site_precision = precision_score(y_site, prediction_site, zero_division=0)
    site_recall = recall_score(y_site, prediction_site, zero_division=0)
    site_f1 = f1_score(y_site, prediction_site, zero_division=0)

    per_site_results.append({
        "site": site,
        "n_images": len(site_results),
        "auc": site_auc,
        "accuracy": site_accuracy,
        "precision": site_precision,
        "recall": site_recall,
        "f1": site_f1,
        "threshold": selected_threshold
    })

per_site_results_df = pd.DataFrame(per_site_results)
display(per_site_results_df)


## 7. Diagnostic evaluation

Confusion matrices, classification reports, ROC curves and score distributions
are used to inspect the type and location of classification errors.

ROC-AUC measures the model's ability to rank blocked images above clear images
across all possible thresholds. Accuracy, precision, recall and F1 are
threshold-dependent and therefore describe performance at the operating point
selected from the validation data.

The additional uncertainty ranking identifies images whose blockage scores lie
closest to the decision threshold. Such cases are useful for qualitative error
inspection because they correspond to the model's least decisive predictions.


In [ ]:
# Pooled + per-site confusion matrices and classification reports

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Display names for figures/reports

DISPLAY_NAMES = {
    "Cornwall_BudeCedarGrove": "Cornwall Bude Cedar Grove",
    "sites_brutondam_cam1": "Brutondam"
}


# Confusion matrix plotting function

def plot_confusion(y_true, y_pred, title, ax):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(title, fontsize=10)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["clear", "blocked"])
    ax.set_yticklabels(["clear", "blocked"])

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    # Add values with adaptive text colour
    for i in range(2):
        for j in range(2):

            rgba = im.cmap(im.norm(cm[i, j]))

            luminance = (
                0.299 * rgba[0]
                + 0.587 * rgba[1]
                + 0.114 * rgba[2]
            )

            text_color = (
                "white" if luminance < 0.5 else "black"
            )

            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color=text_color,
                fontsize=12
            )


# Plot: pooled + individual held-out sites


fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4)
)

fig.suptitle(
    "DINOv3 + MLP",
    fontsize=13
)


# Pooled
plot_confusion(
    test_results["y_true"],
    test_results["predicted_class"],
    "Pooled",
    axes[0]
)


# Individual held-out sites
for ax, site in zip(axes[1:], TEST_SITES):

    site_results = test_results[
        test_results["site"] == site
    ]

    plot_confusion(
        site_results["y_true"],
        site_results["predicted_class"],
        DISPLAY_NAMES.get(site, site),
        ax
    )


plt.tight_layout()
plt.show()



# Pooled classification report

print("\nPOOLED")

print(
    classification_report(
        test_results["y_true"],
        test_results["predicted_class"],
        labels=[0, 1],
        target_names=["clear", "blocked"],
        zero_division=0
    )
)



# Per-site classification reports

for site in TEST_SITES:

    site_results = test_results[
        test_results["site"] == site
    ]

    print(f"\n{DISPLAY_NAMES.get(site, site)}")

    print(
        classification_report(
            site_results["y_true"],
            site_results["predicted_class"],
            labels=[0, 1],
            target_names=["clear", "blocked"],
            zero_division=0
        )
    )


In [ ]:
# ROC curves

plt.figure(figsize=(7, 6))

# Validation ROC
validation_fpr, validation_tpr, _ = roc_curve(
    validation_targets,
    validation_scores
)

plt.plot(
    validation_fpr,
    validation_tpr,
    label=f"Validation AUC {validation_auc:.3f}"
)


# Held-out test sites
for site in TEST_SITES:

    site_results = test_results[
        test_results["site"] == site
    ]

    site_fpr, site_tpr, _ = roc_curve(
        site_results["y_true"],
        site_results["blockage_score"]
    )

    site_auc = roc_auc_score(
        site_results["y_true"],
        site_results["blockage_score"]
    )

    # Use clean display name
    site_name = DISPLAY_NAMES.get(site, site)

    plt.plot(
        site_fpr,
        site_tpr,
        label=f"{site_name} AUC {site_auc:.3f}"
    )


# Random classifier reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle=":",
    color="grey"
)

plt.xlabel("False positive rate")
plt.ylabel("True positive rate")

plt.title(
    "ROC curves, validation and held-out test sites"
)

plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Score distributions per held-out site

for site in TEST_SITES:

    site_results = test_results[
        test_results["site"] == site
    ]

    clear_scores = site_results.loc[
        site_results["y_true"] == 0,
        "blockage_score"
    ]

    blocked_scores = site_results.loc[
        site_results["y_true"] == 1,
        "blockage_score"
    ]

    # Clean site name for display only
    site_name = DISPLAY_NAMES.get(site, site)

    plt.figure(figsize=(7, 5))

    plt.hist(
        clear_scores,
        bins=30,
        alpha=0.6,
        label="Clear"
    )

    plt.hist(
        blocked_scores,
        bins=30,
        alpha=0.6,
        label="Blocked"
    )

    plt.axvline(
        selected_threshold,
        linestyle=":",
        linewidth=2,
        label=f"Threshold {selected_threshold:.4f}"
    )

    plt.xlabel("Blockage score")
    plt.ylabel("Number of images")
    plt.title(f"Score distribution, {site_name}")

    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Error inspection buckets

def assign_bucket(row):
    true_value = row["y_true"]
    predicted_value = row["predicted_class"]

    if true_value == 1 and predicted_value == 1:
        return "TP"
    if true_value == 0 and predicted_value == 0:
        return "TN"
    if true_value == 0 and predicted_value == 1:
        return "FP"
    return "FN"


test_results["bucket"] = test_results.apply(assign_bucket, axis=1)

print("Overall buckets:")
print(test_results["bucket"].value_counts())

print("\nBuckets by site:")
print(test_results.groupby("site")["bucket"].value_counts().unstack(fill_value=0))


In [ ]:
# Uncertainty ranking

test_results["distance_from_threshold"] = np.abs(test_results["blockage_score"] - selected_threshold)

TOP_K = 10

for site in TEST_SITES:
    review_queue = (
        test_results[test_results["site"] == site]
        .sort_values("distance_from_threshold")
        .head(TOP_K)
    )

    print(f"\nMost uncertain images for {site}")
    display(review_queue[[
        "filename", "true_label", "predicted_label",
        "blockage_score", "distance_from_threshold", "bucket"
    ]])


## Additional site-calibration experiment

The raw MLP experiment evaluates if a classifier trained on frozen DINOv3
representations transfers directly to unseen monitoring sites. An additional
calibration experiment is included below to examine whether a small set of
**clear reference images from a new site** can reduce site-specific shifts in
the embedding space.

For each development site, the mean embedding of its clear images is used as a
reference vector and subtracted from the corresponding embeddings. For the two
held-out sites, 20 clear images are sampled to estimate an analogous
site-specific reference vector. These calibration images are excluded from
evaluation.

The calibration procedure is repeated ten times with different seeded samples.
It therefore represents a limited adaptation scenario rather than a strictly
zero-shot cross-site evaluation.


In [ ]:
# Compute clear reference vectors for site calibration

development_site_means = compute_site_means(
    X_fit,
    fit_meta,
    clear_only=True
)

X_fit_calibrated = center_by_site(
    X_fit,
    fit_meta,
    development_site_means
)

X_validation_calibrated = center_by_site(
    X_validation,
    validation_meta,
    development_site_means
)

print("Development site reference means computed from clear fit images:")
for site in development_site_means:
    print(f"  {site}")


In [ ]:
# Retrain MLP on calibrated development data
# using fixed best hyperparameters

best_configuration_number = int(
    best_result["configuration"]
)

(
    best_hidden_dims,
    best_dropout,
    best_learning_rate,
    best_weight_decay
) = parameter_combinations[
    best_configuration_number - 1
]

print("Reusing best configuration:")
print(f"  hidden_dims: {best_hidden_dims}")
print(f"  dropout: {best_dropout}")
print(f"  learning_rate: {best_learning_rate}")
print(f"  weight_decay: {best_weight_decay}")


calibrated_training_result = train_mlp(
    X_fit=X_fit_calibrated,
    y_fit=y_fit,
    X_validation=X_validation_calibrated,
    y_validation=y_validation,
    hidden_dims=best_hidden_dims,
    dropout=best_dropout,
    learning_rate=best_learning_rate,
    weight_decay=best_weight_decay,
    batch_size=64,
    max_epochs=100,
    patience=8,
    seed=RANDOM_SEED
)

calibrated_model = calibrated_training_result["model"]

print(
    f"\nCalibrated validation AUC: "
    f"{calibrated_training_result['best_validation_auc']:.4f}"
)


In [ ]:
# Select threshold on calibrated validation scores

validation_loader_calibrated = create_loader(
    X_validation_calibrated,
    y_validation,
    batch_size=64,
    shuffle=False
)

validation_scores_calibrated, validation_targets_calibrated = (
    predict_scores(
        calibrated_model,
        validation_loader_calibrated
    )
)

precision_values_c, recall_values_c, thresholds_c = (
    precision_recall_curve(
        validation_targets_calibrated,
        validation_scores_calibrated
    )
)

f1_values_c = (
    2 * precision_values_c * recall_values_c
    / (
        precision_values_c
        + recall_values_c
        + 1e-12
    )
)

valid_f1_values_c = f1_values_c[:-1]

best_threshold_index_c = np.argmax(
    valid_f1_values_c
)

selected_threshold_calibrated = thresholds_c[
    best_threshold_index_c
]

print(
    f"Selected calibrated threshold: "
    f"{selected_threshold_calibrated:.6f}"
)


In [ ]:
# Site calibration repeats using clear reference images

CALIBRATION_SIZE = 20
CALIBRATION_REPEATS = 10

calibrated_test_records = []
calibrated_detailed_predictions = {}

for repeat in range(1, CALIBRATION_REPEATS + 1):

    repeat_rng = np.random.default_rng(
        RANDOM_SEED + repeat
    )

    test_site_reference_means = {}
    calibration_index_set = set()


    # Select clear calibration images for each held-out site

    for site in TEST_SITES:

        clear_site_mask = (
            (test_meta["site"] == site)
            & (test_meta["label"].str.lower() == "clear")
        ).to_numpy()

        clear_site_indices = np.where(
            clear_site_mask
        )[0]

        calibration_indices = repeat_rng.choice(
            clear_site_indices,
            size=CALIBRATION_SIZE,
            replace=False
        )

        # Mean embedding of the 20 clear calibration images
        test_site_reference_means[site] = (
            X_test[calibration_indices].mean(axis=0)
        )

        calibration_index_set.update(
            calibration_indices.tolist()
        )


    # Calibrate test embeddings using each site's
    # clear reference mean


    X_test_calibrated = center_by_site(
        X_test,
        test_meta,
        test_site_reference_means
    )


    # Exclude calibration images from evaluation

    evaluation_mask = np.ones(
        len(X_test),
        dtype=bool
    )

    evaluation_mask[
        list(calibration_index_set)
    ] = False

    X_eval = X_test_calibrated[
        evaluation_mask
    ]

    y_eval = y_test[
        evaluation_mask
    ]

    meta_eval = (
        test_meta
        .loc[evaluation_mask]
        .reset_index(drop=True)
    )


    # Predict using calibrated model

    eval_loader = create_loader(
        X_eval,
        y_eval,
        batch_size=64,
        shuffle=False
    )

    calibration_scores, calibration_targets = (
        predict_scores(
            calibrated_model,
            eval_loader
        )
    )

    calibration_predictions = (
        calibration_scores
        >= selected_threshold_calibrated
    ).astype(int)


    # Pooled metrics

    pooled_auc = roc_auc_score(
        calibration_targets,
        calibration_scores
    )

    pooled_accuracy = accuracy_score(
        calibration_targets,
        calibration_predictions
    )

    pooled_precision = precision_score(
        calibration_targets,
        calibration_predictions,
        zero_division=0
    )

    pooled_recall = recall_score(
        calibration_targets,
        calibration_predictions,
        zero_division=0
    )

    pooled_f1 = f1_score(
        calibration_targets,
        calibration_predictions,
        zero_division=0
    )

    calibrated_test_records.append({
        "repeat": repeat,
        "site": "pooled",
        "auc": pooled_auc,
        "accuracy": pooled_accuracy,
        "precision": pooled_precision,
        "recall": pooled_recall,
        "f1": pooled_f1
    })


    # Per-site metrics

    for site in TEST_SITES:

        site_mask = (
            meta_eval["site"] == site
        ).to_numpy()

        site_auc = roc_auc_score(
            calibration_targets[site_mask],
            calibration_scores[site_mask]
        )

        site_accuracy = accuracy_score(
            calibration_targets[site_mask],
            calibration_predictions[site_mask]
        )

        site_precision = precision_score(
            calibration_targets[site_mask],
            calibration_predictions[site_mask],
            zero_division=0
        )

        site_recall = recall_score(
            calibration_targets[site_mask],
            calibration_predictions[site_mask],
            zero_division=0
        )

        site_f1 = f1_score(
            calibration_targets[site_mask],
            calibration_predictions[site_mask],
            zero_division=0
        )

        calibrated_test_records.append({
            "repeat": repeat,
            "site": site,
            "auc": site_auc,
            "accuracy": site_accuracy,
            "precision": site_precision,
            "recall": site_recall,
            "f1": site_f1
        })


    # Store detailed predictions for later figures


    calibrated_detailed_predictions[repeat] = {
        "meta": meta_eval,
        "scores": calibration_scores,
        "targets": calibration_targets,
        "predictions": calibration_predictions
    }



# Combine results from all calibration repeats

calibrated_test_results_df = pd.DataFrame(
    calibrated_test_records
)

display(calibrated_test_results_df)


In [ ]:
# Confusion matrices for calibrated test results
# Displays one calibration repeat for visualisation

display_repeat = min(
    calibrated_detailed_predictions.keys()
)

display_data = calibrated_detailed_predictions[
    display_repeat
]

display_meta = display_data["meta"]
display_targets = display_data["targets"]
display_predictions = display_data["predictions"]


for site in TEST_SITES:

    site_mask = (
        display_meta["site"] == site
    ).to_numpy()

    site_targets = display_targets[
        site_mask
    ]

    site_predictions = display_predictions[
        site_mask
    ]

    matrix = confusion_matrix(
        site_targets,
        site_predictions,
        labels=[0, 1]
    )

    # Clean display name
    site_name = DISPLAY_NAMES.get(
        site,
        site
    )

    figure, axis = plt.subplots(
        figsize=(5, 4)
    )

    image = axis.imshow(
        matrix,
        cmap="Blues"
    )

    # Figure title
    axis.set_title(
        f"{site_name}, calibrated",
        fontsize=10
    )

    axis.set_xticks([0, 1])
    axis.set_yticks([0, 1])

    axis.set_xticklabels(
        ["clear", "blocked"]
    )

    axis.set_yticklabels(
        ["clear", "blocked"]
    )

    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")

    # Add values with adaptive text colour
    for row in range(2):
        for column in range(2):

            rgba = image.cmap(
                image.norm(
                    matrix[row, column]
                )
            )

            luminance = (
                0.299 * rgba[0]
                + 0.587 * rgba[1]
                + 0.114 * rgba[2]
            )

            text_color = (
                "white"
                if luminance < 0.5
                else "black"
            )

            axis.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
                color=text_color,
                fontsize=12
            )

    figure.tight_layout()
    plt.show()

    # Classification report
    print(
        f"\n{site_name}, calibrated"
    )

    print(
        classification_report(
            site_targets,
            site_predictions,
            labels=[0, 1],
            target_names=[
                "clear",
                "blocked"
            ],
            zero_division=0
        )
    )


Confusion matrix from one of the 10 calibration repeats.

In [ ]:
# Aggregate calibration repeats into mean/std summary per site

calibrated_summary = (
    calibrated_test_results_df
    .groupby("site")[["auc", "accuracy", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)

# Clean site names for display
calibrated_summary = calibrated_summary.rename(
    index={
        "pooled": "Pooled",
        "Cornwall_BudeCedarGrove": "Cornwall Bude Cedar Grove",
        "sites_brutondam_cam1": "Brutondam"
    }
)

display(calibrated_summary)


## 9. Save reproducibility artefacts

The final cells export model-selection results, per-site metrics, image-level
scores, training history and trained model parameters. These artefacts allow
the reported evaluation to be reconstructed without relying on notebook
display outputs.


In [ ]:
# Save all outputs, raw and calibrated

os.makedirs(RESULTS_FOLDER, exist_ok=True)


# Raw MLP results

grid_results_df.to_csv(
    RESULTS_FOLDER / "mlp_grid_results.csv",
    index=False
)

per_site_results_df.to_csv(
    RESULTS_FOLDER / "mlp_per_site_results.csv",
    index=False
)

test_results.to_csv(
    RESULTS_FOLDER / "mlp_scored_test_set.csv",
    index=False
)

best_history.to_csv(
    RESULTS_FOLDER / "mlp_best_training_history.csv",
    index=False
)

torch.save(
    best_model.state_dict(),
    RESULTS_FOLDER / "mlp_best_model_state.pt"
)


# Calibrated MLP results


calibrated_test_results_df.to_csv(
    RESULTS_FOLDER / "mlp_calibrated_test_repeats.csv",
    index=False
)

calibrated_summary.to_csv(
    RESULTS_FOLDER / "mlp_calibrated_summary.csv"
)

torch.save(
    calibrated_model.state_dict(),
    RESULTS_FOLDER / "mlp_calibrated_model_state.pt"
)



# Calibrated model configuration


with open(
    RESULTS_FOLDER / "mlp_calibrated_best_config.json",
    "w"
) as f:

    json.dump(
        {
            "hidden_dims": best_hidden_dims,
            "dropout": best_dropout,
            "learning_rate": best_learning_rate,
            "weight_decay": best_weight_decay,
            "selected_threshold_raw": float(
                selected_threshold
            ),
            "selected_threshold_calibrated": float(
                selected_threshold_calibrated
            ),
            "calibration_size": CALIBRATION_SIZE,
            "calibration_repeats": CALIBRATION_REPEATS
        },
        f,
        indent=2
    )


print("Results saved to:")
print(RESULTS_FOLDER)


In [ ]:
# Raw vs calibrated confusion matrices
# Pooled + held-out sites

# Use one stored calibration repeat for the calibrated confusion matrices
display_repeat = min(calibrated_detailed_predictions.keys())

calibrated_data = calibrated_detailed_predictions[display_repeat]

calibrated_meta = calibrated_data["meta"]
calibrated_targets = calibrated_data["targets"]
calibrated_predictions = calibrated_data["predictions"]


# Confusion matrix plotting function
def plot_final_confusion(y_true, y_pred, title, ax):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    im = ax.imshow(
        cm,
        cmap="Blues"
    )

    ax.set_title(
        title,
        fontsize=10
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(
        ["clear", "blocked"]
    )

    ax.set_yticklabels(
        ["clear", "blocked"]
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    # Add values with adaptive text colour
    for row in range(2):
        for column in range(2):

            rgba = im.cmap(
                im.norm(cm[row, column])
            )

            luminance = (
                0.299 * rgba[0]
                + 0.587 * rgba[1]
                + 0.114 * rgba[2]
            )

            text_color = (
                "white"
                if luminance < 0.5
                else "black"
            )

            ax.text(
                column,
                row,
                str(cm[row, column]),
                ha="center",
                va="center",
                color=text_color,
                fontsize=12
            )



# Create 2 x 3 comparison figure

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 8)
)



# ROW 1: RAW

# Raw pooled
plot_final_confusion(
    test_results["y_true"],
    test_results["predicted_class"],
    "Pooled",
    axes[0, 0]
)


# Raw individual sites
for column, site in enumerate(TEST_SITES, start=1):

    site_results = test_results[
        test_results["site"] == site
    ]

    site_name = DISPLAY_NAMES.get(
        site,
        site
    )

    plot_final_confusion(
        site_results["y_true"],
        site_results["predicted_class"],
        site_name,
        axes[0, column]
    )



# ROW 2: CALIBRATED

# Calibrated pooled
plot_final_confusion(
    calibrated_targets,
    calibrated_predictions,
    "Pooled",
    axes[1, 0]
)


# Calibrated individual sites
for column, site in enumerate(TEST_SITES, start=1):

    site_mask = (
        calibrated_meta["site"] == site
    ).to_numpy()

    site_name = DISPLAY_NAMES.get(
        site,
        site
    )

    plot_final_confusion(
        calibrated_targets[site_mask],
        calibrated_predictions[site_mask],
        site_name,
        axes[1, column]
    )



# Row labels

axes[0, 0].text(
    -0.55,
    0.5,
    "Raw",
    transform=axes[0, 0].transAxes,
    rotation=90,
    va="center",
    ha="center",
    fontsize=12,
    fontweight="bold"
)

axes[1, 0].text(
    -0.55,
    0.5,
    "Calibrated",
    transform=axes[1, 0].transAxes,
    rotation=90,
    va="center",
    ha="center",
    fontsize=12,
    fontweight="bold"
)


fig.suptitle(
    "DINOv3 + MLP: Raw vs Calibrated Test Performance",
    fontsize=14
)

plt.tight_layout()
plt.show()


### Interpretation of raw and calibrated results

In the recorded dissertation run, the raw DINOv3 + MLP classifier transferred
unevenly across the two held-out monitoring sites. Performance was substantially
stronger at Brutondam, whereas Cornwall Bude Cedar Grove showed a pronounced
false-positive tendency: many clear images received blockage scores above the
validation-selected threshold.

The calibration experiment reduced this false-positive tendency at Cornwall
Bude Cedar Grove by centring embeddings relative to clear reference images from
the site. However, this improvement was accompanied by lower blockage recall.
At Brutondam, where the uncalibrated classifier was already strong, calibration
provided little benefit and increased the number of missed blocked images.

These results indicate that site-specific calibration can alter the
precision-recall trade-off, but does not uniformly improve cross-site
generalisation. The raw and calibrated evaluations should therefore be
interpreted as different deployment assumptions: the raw model requires no
labelled reference images from a new site, whereas the calibrated model uses a
small set of known-clear images.
